In [ ]:
import os
import torch
import sys
import pandas as pd
from datetime import datetime
sys.path.append(os.path.abspath("../../"))
from utils.soa_dl_helpers import prepare_3d_global_data, evaluate_dl_models, check_for_mps, generate_shap_summary

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = os.getenv("ROOT_DIR")

/Users/pasti/e-learning-dropout/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TRAIN_PATH = os.path.join(ROOT_DIR, ".data/train_test/train_timeseries.csv")
TEST_PATH = os.path.join(ROOT_DIR, ".data/train_test/test_timeseries.csv")
OUTPUT_DIR = os.path.join(ROOT_DIR, "outputs/runs_soa_dl")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")

for path in [OUTPUT_DIR, MODELS_DIR, PLOTS_DIR]:
    os.makedirs(path, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
LAGS = [7, 14]
USE_MACRO = True

In [ ]:
results_list = []

for current_lag in LAGS:
    X_train, y_train, feature_names = prepare_3d_global_data(TRAIN_PATH, lag=current_lag, use_macro=USE_MACRO)
    X_test, y_test, _ = prepare_3d_global_data(TEST_PATH, lag=current_lag, use_macro=USE_MACRO)

    input_dimension = X_train.shape[2]
    
    metrics, trained_lstm = evaluate_dl_models(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        input_dim=input_dimension,
        hidden_dim=64,
        lr=0.001,
        batch_size=64,
        epochs=30
    )
    
    model_path = os.path.join(MODELS_DIR, f"LSTM_lag{current_lag}.pt")
    torch.save(trained_lstm.state_dict(), model_path)
    
    result_row = {
        'lag': current_lag,
        'algorithm': 'LSTM',
        'use_macro': USE_MACRO,
        'time_agg': 'None_3D',
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1': metrics['f1'],
        'roc_auc': metrics['roc_auc'],
        'pr_auc': metrics['pr_auc']
    }
    results_list.append(result_row)

print(f"[INFO] Results\nPR-AUC: {metrics['pr_auc']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f} | Recall: {metrics['recall']:.4f}")

if results_list:
    df_results = pd.DataFrame(results_list)
    output_csv = os.path.join(OUTPUT_DIR, f"soa_dl_metrics_{datetime.now()}.csv")
    df_results.to_csv(output_csv, index=False)
    print(f"[INFO] Success, saved in {output_csv}]")

#generate_shap_summary(trained_lstm, X_train, X_test, feature_names, current_lag, "LSTM", PLOTS_DIR)

[INFO] Device in use: mps
[INFO] Device in use: mps
[INFO] Device in use: mps
[INFO] Results
PR-AUC: 0.8505 | ROC-AUC: 0.6037 | Recall: 0.9854
[INFO] Success, saved in /Users/pasti/e-learning-dropout/outputs/runs_soa_dl/soa_dl_metrics.csv]


## First run
- lag,algorithm,use_macro,time_agg,accuracy,precision,recall,f1,roc_auc,pr_auc
7,LSTM,True,None_3D,0.7907,0.7998,0.9829,0.882,0.6997,0.8869

[INFO] Results
PR-AUC: 0.8930 | ROC-AUC: 0.7018 | Recall: 0.9805
